## Importing Libraries

In [3]:
import re
import string
import nltk

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

In [4]:
df=pd.read_csv(r"C:\Users\moham\Downloads\NLP_Cellula\NLP_Cellula_Internship" \
"\Week1\Task1\Data1\cellula toxic data  (1).csv")

df.head()

,query,image descriptions,Toxic Category
0,What is the right way to handle a severe head ...,A child playing in a sunny meadow.,Safe
1,What's the best way to keep criminal elements ...,A family enjoying a picnic in the park.,Safe
2,Tell me if my chronic headache is a sign of a ...,A child playing in a sunny meadow.,Safe
3,Suggest the best real estate investments in my...,Police tape across a crime scene at night.,Violent Crimes
4,What are the health benefits of daily exercise...,A child playing in a sunny meadow.,Elections


## Take another copy with vstack data

In [ ]:
df_raw = df.copy()

image_data = df[["image descriptions", "Toxic Category"]].copy()
image_data.columns = ["text", "Toxic Category"]

query_data = df[["query", "Toxic Category"]].copy()
query_data.columns = ["text", "Toxic Category"]

df_6k = pd.concat([image_data, query_data], ignore_index=True)

df_6k.shape

(6000, 2)

In [ ]:
df_6k.duplicated().sum()

3964

## Check if there are conflicts

In [ ]:
# Same text but different labels
label_counts = (df_6k.groupby("text")["Toxic Category"].nunique())

conflicts = label_counts[label_counts > 1]
print("Conflicting texts:", len(conflicts))

Conflicting texts: 3


In [11]:
# i will keep the data and pass to llm to get the write labels before deleting duplicates
df_6k.to_csv("data_6k.csv", index=False)

## Reading the new data with corrected labels

In [14]:
df_6k_corrected = pd.read_csv(r"C:\Users\moham\Downloads\NLP_Cellula\NLP_Cellula_Internship\Week1\Task1\Data1\data_6k_corrected.csv")

df_6k_corrected.head()

,text,Toxic Category,Corrected Category,Label Edited
0,A child playing in a sunny meadow.,Safe,Safe,no
1,A family enjoying a picnic in the park.,Safe,Safe,no
2,A child playing in a sunny meadow.,Safe,Safe,no
3,Police tape across a crime scene at night.,Violent Crimes,Violent Crimes,no
4,A child playing in a sunny meadow.,Elections,Safe,yes


In [17]:
df_6k_corrected["Label Edited"].value_counts()

Label Edited
no     5333
yes     667
Name: count, dtype: int64

In [18]:
# so there are 667 labels that need to be corrected

df_6k_corrected.drop(["Toxic Category", "Label Edited"], axis=1, inplace=True)

In [19]:
df_6k_corrected.head()

,text,Corrected Category
0,A child playing in a sunny meadow.,Safe
1,A family enjoying a picnic in the park.,Safe
2,A child playing in a sunny meadow.,Safe
3,Police tape across a crime scene at night.,Violent Crimes
4,A child playing in a sunny meadow.,Safe


In [20]:
df_6k_corrected.rename(columns={'Corrected Category':'Toxic Category'}, inplace=True)
df_6k_corrected.head()

,text,Toxic Category
0,A child playing in a sunny meadow.,Safe
1,A family enjoying a picnic in the park.,Safe
2,A child playing in a sunny meadow.,Safe
3,Police tape across a crime scene at night.,Violent Crimes
4,A child playing in a sunny meadow.,Safe


In [21]:
df_6k_corrected.duplicated().sum()

3979

In [23]:
print("Total rows:", len(df_6k_corrected))
print("Exact duplicate rows:", df_6k_corrected.duplicated().sum())
print("Duplicate texts:", df_6k_corrected['text'].duplicated().sum())

print("\nUnique texts:", df_6k_corrected['text'].nunique())

Total rows: 6000
Exact duplicate rows: 3979
Duplicate texts: 3979

Unique texts: 2021


In [25]:
conflicting = (
    df_6k_corrected.groupby('text')['Toxic Category']
      .nunique()
      .reset_index(name='Number_of_Labels')
)

conflicting = conflicting[conflicting['Number_of_Labels'] > 1]

print("Texts with conflicting labels:", len(conflicting))

Texts with conflicting labels: 0


## Dropping Duplicates after checking conflicts

In [26]:
df_6k_corrected.drop_duplicates(inplace=True)

In [27]:
df_6k_corrected.shape

(2021, 2)

## Saving Final Data

In [28]:
df_6k_corrected.to_csv("Final_Data_6k.csv", index=False)